<!-- notebook-header -->
# Tutorial: Classificacao do Zero com NumPy

**Modulo:** 03 - Machine Learning  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Fluxo completo de classificacao, da preparacao dos dados a avaliacao de modelos.


# Tutorial Completo: Classificacao em Machine Learning

**Implementacao 100% do Zero com NumPy**

Um guia passo a passo para construir modelos de classificacao desde o inicio.

Neste tutorial voce aprenderá:
- Como gerar e explorar dados
- Preprocessamento e normalizacao
- Implementar modelos do zero: Logistic Regression, KNN
- Calcular metricas e validar modelos
- Tuning de hiperparametros
- Comparar decisoes de diferentes modelos


---
## 1. Introducao e Configuracao

### O que eh Classificacao?
Classificacao eh a tarefa de prever uma label discreta para um novo exemplo.

### Neste tutorial:
- Dataset: Sintetico com 2 classes
- Features: 2 ou 4 dimensoes (numericas)
- Modelos: Logistic Regression, KNN (implementados do zero)
- Metricas: Accuracy, Precision, Recall, F1, Confusion Matrix
- Validacao: Cross-validation manual
- Tuning: Grid search manual

### Conexao com aprendizado estrategia:
Ao implementar do zero, voce entende cada passo do processo ML.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Configurar matplotlib
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

print('Libs carregadas com sucesso!')
print(f'NumPy versao: {np.__version__}')


---
## 2. Gerar Dataset Sintetico de Classificacao

### O que observar:
- Usamos `np.random.randn()` para distribuicao normal
- Separamos 2 classes com centros diferentes
- Adicionamos ruido (noise) para tornar interessante
- As amostras sao embaralhadas para evitar bias temporal

### O que concluir:
- Dados bem separados sao faceis de classificar
- Ruido torna o problema realista
- A qualidade do dataset afeta muito o modelo

### Conexao com mundo real:
Dados sinteticos sao uteis para testes, mas dados reais tem muito mais complexidade.


In [ ]:
def gerar_dataset_classificacao(n_amostras=200, n_features=2, ruido=0.5, seed=42):
    """Gera dataset sintetico para classificacao binaria.
    
    O que observar: A separacao entre classes e o tamanho do ruido
    O que concluir: Maior ruido = problema mais dificil
    
    Args:
        n_amostras: total de amostras por classe
        n_features: numero de features
        ruido: desvio padrao do ruido gaussiano
        seed: para reproducibilidade
    
    Returns:
        X: array (n_amostras*2, n_features)
        y: array (n_amostras*2,) com labels 0 e 1
    """
    np.random.seed(seed)
    
    # Classe 0: centrada em [-1, -1]
    X_class0 = np.random.randn(n_amostras, n_features) + np.array([-1.5] * n_features)
    X_class0 += np.random.randn(n_amostras, n_features) * ruido
    
    # Classe 1: centrada em [1, 1]
    X_class1 = np.random.randn(n_amostras, n_features) + np.array([1.5] * n_features)
    X_class1 += np.random.randn(n_amostras, n_features) * ruido
    
    # Concatenar
    X = np.vstack([X_class0, X_class1])
    y = np.hstack([np.zeros(n_amostras), np.ones(n_amostras)])
    
    # Embaralhar
    indices = np.random.permutation(len(X))
    return X[indices], y[indices]

# Gerar dataset
X, y = gerar_dataset_classificacao(n_amostras=150, n_features=2, ruido=0.6)

print('Dataset gerado com sucesso!')
print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')
print(f'Classes: {np.unique(y)}')
print(f'Distribuicao: Classe 0: {np.sum(y==0)}, Classe 1: {np.sum(y==1)}')


---
## 3. Analise Exploratoria (EDA)

### O que observar:
- Distribuicao das features por classe
- Separabilidade entre classes no espaco
- Outliers e escalas das features
- Balanceamento das classes

### O que concluir:
- As classes sao bem separadas no espaco 2D
- Nao ha outliers extremos
- As escalas sao diferentes (X varia ~[-3,3])
- As classes estao balanceadas (150 cada)

### Conexao com ML:
A visualizacao ajuda a escolher qual algoritmo usar.

### Por que em ML:
EDA evita surpresas desagradaveis durante o treinamento.


In [ ]:
# Estatisticas descritivas
print('='*60)
print('ESTATISTICAS DESCRITIVAS')
print('='*60)
print(f'\nMedia de X: {np.mean(X, axis=0)}')
print(f'Desvio padrao de X: {np.std(X, axis=0)}')
print(f'Minimo de X: {np.min(X, axis=0)}')
print(f'Maximo de X: {np.max(X, axis=0)}')

print(f'\nMedia por classe:')
for label in [0, 1]:
    media = np.mean(X[y==label], axis=0)
    desvio = np.std(X[y==label], axis=0)
    print(f'  Classe {int(label)}: media={media}, desvio={desvio}')

print(f'\nO que observar: As medias das classes sao bem diferentes!')


In [ ]:
# Visualizacao do dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax = axes[0]
ax.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Classe 0', alpha=0.6, s=50)
ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Classe 1', alpha=0.6, s=50)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Dataset de Classificacao (Espaco 2D)')
ax.legend()
ax.grid(True, alpha=0.3)

# Distribuicao de features
ax = axes[1]
ax.hist(X[y==0, 0], bins=20, alpha=0.5, label='Classe 0, Feature 1', color='red')
ax.hist(X[y==1, 0], bins=20, alpha=0.5, label='Classe 1, Feature 1', color='blue')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Frequencia')
ax.set_title('Distribuicao da Feature 1 por Classe')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.close()

print('O que observar: As classes sao bem separaveis!')
print('Conexao com separabilidade: Dados separaveis sao mais faceis para LR.')


---
## 4. Preprocessamento e Normalizacao

### O que observar:
- Normalizacao (0-1) vs Padronizacao (media=0, std=1)
- A importancia de aplicar transformacoes no treino apenas
- Data leakage pode acontecer aqui facilmente

### O que concluir:
- Padronizacao eh melhor para modelos baseados em distancia (KNN, LR)
- Treino e teste devem usar a MESMA transformacao
- Nunca fit o preprocessador no treino+teste junto

### Por que em ML:
Features com escalas diferentes podem dominar o modelo.

### Por que em ML (escala):
KNN usa distancia euclidiana, que eh sensivel a escala.


In [ ]:
def split_treino_teste(X, y, fracao_teste=0.3, seed=42):
    """Divide dados em treino e teste mantendo proporcoes de classes.
    
    O que observar: O embaralhamento eh importante para evitar bias.
    O que concluir: Split estratificado manteria as proporcoes perfeitas.
    """
    np.random.seed(seed)
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    
    ponto_divisao = int(len(X) * (1 - fracao_teste))
    
    train_indices = indices[:ponto_divisao]
    test_indices = indices[ponto_divisao:]
    
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

X_train, X_test, y_train, y_test = split_treino_teste(X, y, fracao_teste=0.3)

print(f'Tamanho treino: {X_train.shape[0]}')
print(f'Tamanho teste: {X_test.shape[0]}')
print(f'Proporcao treino: {X_train.shape[0] / (X_train.shape[0] + X_test.shape[0]):.2%}')

print(f'\nO que observar: Os dados foram aleatoriamente distribuidos.')


In [ ]:
def padronizar(X, media=None, desvio=None):
    """Padroniza os dados (z-score normalization).
    
    Se media e desvio nao forem fornecidos, calcula-os a partir de X.
    Caso contrario, usa os fornecidos (para teste).
    
    O que observar: Esta funcao implementa (X - media) / desvio
    O que concluir: Padronizacao muda a distribuicao para N(0,1)
    Por que em ML: Melhora convergencia do gradient descent
    """
    if media is None:
        media = np.mean(X, axis=0)
    if desvio is None:
        desvio = np.std(X, axis=0)
    
    # Evitar divisao por zero
    desvio[desvio == 0] = 1.0
    
    return (X - media) / desvio, media, desvio

# Padronizar (usar parametros do treino para teste)
X_train_norm, media_treino, desvio_treino = padronizar(X_train)
X_test_norm, _, _ = padronizar(X_test, media_treino, desvio_treino)

print('Padronizacao executada!')
print(f'Media de X_train normalizado: {np.mean(X_train_norm, axis=0)}')
print(f'Desvio padrao de X_train normalizado: {np.std(X_train_norm, axis=0)}')

print(f'\nMedia de X_test normalizado: {np.mean(X_test_norm, axis=0)}')
print(f'Desvio padrao de X_test normalizado: {np.std(X_test_norm, axis=0)}')

print('\nO que concluir: X_test nao tem media 0 pq usou params do treino!')


---
## 5. Metricas de Classificacao

### O que observar:
- Confusion Matrix descreve erros e acertos (TP, TN, FP, FN)
- Accuracy pode ser enganoso em dados desbalanceados
- Precision vs Recall tem tradeoff importante
- F1 balanceia ambos (media harmonica)

### O que concluir:
- F1 eh util quando F+ ou F- sao igualmente importantes
- Precision importa quando falsos positivos sao caros
- Recall importa quando falsos negativos sao caros

### Por que em ML:
Metricas diferentes revelam diferentes aspectos do desempenho.

### Conexao com negocio:
A escolha de metrica deve refletir o custo real dos erros.


In [ ]:
def confusion_matrix(y_true, y_pred):
    """Calcula matriz de confusao para classificacao binaria.
    
    Retorna: [[TN, FP],
               [FN, TP]]
    """
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])

def accuracy(y_true, y_pred):
    """Acuracia = (TP + TN) / Total.
    O que eh: Proporcao de predicoes corretas
    """
    return np.sum(y_true == y_pred) / len(y_true)

def precision(y_true, y_pred):
    """Precisao = TP / (TP + FP).
    O que eh: De todos que predissemos como positivos, quantos estavam certos?
    """
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)

def recall(y_true, y_pred):
    """Recall = TP / (TP + FN).
    O que eh: De todos os positivos reais, quantos acertamos?
    """
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)

def f1_score(y_true, y_pred):
    """F1 = 2 * (Precision * Recall) / (Precision + Recall).
    O que eh: Media harmonica de Precision e Recall
    """
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    if prec + rec == 0:
        return 0.0
    return 2 * (prec * rec) / (prec + rec)

def calcular_metricas(y_true, y_pred):
    """Calcula todas as metricas basicas."""
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy(y_true, y_pred)
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return {
        'confusion_matrix': cm,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    }

print('Funcoes de metricas definidas com sucesso!')


---
## 6. Modelo 1: Logistic Regression (do Zero)

### O que observar:
- Usa sigmoid(wTx + b) para probabilidades
- Otimizacao com gradient descent
- Funcao loss: binary cross-entropy
- Regularizacao L2 para evitar overfitting

### O que concluir:
- LR eh simples mas poderoso para problemas linearmente separaveis
- Regularizacao previne overfitting
- Gradientes guiam o aprendizado na direcao certa

### Conexao com redes neurais:
Redes neurais usam conceito similar (sigmoid, backprop).

### Por que em ML:
LR eh base para muitos algoritmos mais complexos.

### Por que gradient descent:
Eh o algoritmo mais usado em ML moderno.


In [ ]:
class LogisticRegression:
    """Regressao Logistica implementada do zero com NumPy.
    
    O que observar:
    - sigmoid transforma valores em [0, 1]
    - Gradient descent atualiza pesos iterativamente
    - Cross-entropy penaliza predicoes confiantes mas erradas
    """
    
    def __init__(self, learning_rate=0.01, n_iteracoes=1000, regularizacao=0.0):
        self.learning_rate = learning_rate
        self.n_iteracoes = n_iteracoes
        self.regularizacao = regularizacao
        self.pesos = None
        self.bias = None
        self.historico_loss = []
    
    def sigmoid(self, z):
        """Funcao sigmoid: 1 / (1 + e^(-z)).
        O que observar: Transforma qualquer valor em [0, 1]
        """
        z = np.clip(z, -500, 500)  # Evitar overflow
        return 1 / (1 + np.exp(-z))
    
    def calcular_loss(self, X, y, y_pred):
        """Binary cross-entropy loss.
        O que observar: Penaliza ambos FP e FN
        O que concluir: Loss diminui durante treinamento
        """
        m = len(y)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
        
        # Adicionar regularizacao L2
        if self.regularizacao > 0:
            loss += self.regularizacao * np.sum(self.pesos ** 2) / (2 * m)
        
        return loss
    
    def treinar(self, X, y):
        """Treina o modelo com gradient descent.
        O que observar: Atualiza pesos em direcao do gradiente negativo
        """
        m, n = X.shape
        
        # Inicializar pesos e bias
        self.pesos = np.zeros(n)
        self.bias = 0
        self.historico_loss = []
        
        for iteracao in range(self.n_iteracoes):
            # Forward pass
            z = np.dot(X, self.pesos) + self.bias
            y_pred = self.sigmoid(z)
            
            # Backward pass (calcular gradientes)
            erro = y_pred - y
            gradiente_w = np.dot(X.T, erro) / m
            gradiente_b = np.mean(erro)
            
            # Adicionar regularizacao ao gradiente
            if self.regularizacao > 0:
                gradiente_w += self.regularizacao * self.pesos / m
            
            # Atualizar parametros
            self.pesos -= self.learning_rate * gradiente_w
            self.bias -= self.learning_rate * gradiente_b
            
            # Guardar loss
            loss = self.calcular_loss(X, y, y_pred)
            self.historico_loss.append(loss)
        
        return self
    
    def prever_proba(self, X):
        """Preve probabilidades da classe 1."""
        z = np.dot(X, self.pesos) + self.bias
        return self.sigmoid(z)
    
    def prever(self, X, threshold=0.5):
        """Preve classes (0 ou 1).
        O que observar: Usa threshold 0.5 por padrao
        """
        return (self.prever_proba(X) >= threshold).astype(int)

# Treinar modelo
lr = LogisticRegression(learning_rate=0.1, n_iteracoes=1000, regularizacao=0.01)
lr.treinar(X_train_norm, y_train)

print(f'Treinamento completo!')
print(f'Pesos aprendidos: {lr.pesos}')
print(f'Bias aprendido: {lr.bias}')


In [ ]:
# Avaliar modelo LR
y_train_pred_lr = lr.prever(X_train_norm)
y_test_pred_lr = lr.prever(X_test_norm)

metricas_train_lr = calcular_metricas(y_train, y_train_pred_lr)
metricas_test_lr = calcular_metricas(y_test, y_test_pred_lr)

print('='*60)
print('LOGISTIC REGRESSION - METRICAS')
print('='*60)
print(f'\nTREINO:')
print(f'  Accuracy:  {metricas_train_lr["accuracy"]:.4f}')
print(f'  Precision: {metricas_train_lr["precision"]:.4f}')
print(f'  Recall:    {metricas_train_lr["recall"]:.4f}')
print(f'  F1:        {metricas_train_lr["f1"]:.4f}')

print(f'\nTESTE:')
print(f'  Accuracy:  {metricas_test_lr["accuracy"]:.4f}')
print(f'  Precision: {metricas_test_lr["precision"]:.4f}')
print(f'  Recall:    {metricas_test_lr["recall"]:.4f}')
print(f'  F1:        {metricas_test_lr["f1"]:.4f}')

print(f'\nCONFUSION MATRIX (TESTE):')
cm = metricas_test_lr['confusion_matrix']
print(f'  TN={cm[0,0]:.0f}, FP={cm[0,1]:.0f}')
print(f'  FN={cm[1,0]:.0f}, TP={cm[1,1]:.0f}')

print('\nO que observar: Diferenca entre treino e teste indica overfitting.')


In [ ]:
# Visualizar convergencia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss durante treinamento
ax = axes[0]
ax.plot(lr.historico_loss, linewidth=2)
ax.set_xlabel('Iteracao')
ax.set_ylabel('Loss (Binary Cross-Entropy)')
ax.set_title('Convergencia do Logistic Regression')
ax.grid(True, alpha=0.3)

# Decisao boundary
ax = axes[1]
h = 0.02
x_min, x_max = X_train_norm[:, 0].min() - 0.5, X_train_norm[:, 0].max() + 0.5
y_min, y_max = X_train_norm[:, 1].min() - 0.5, X_train_norm[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = lr.prever(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=1, alpha=0.3, colors=['red', 'blue'])
ax.scatter(X_train_norm[y_train==0, 0], X_train_norm[y_train==0, 1], c='red', label='Classe 0', s=30, alpha=0.7)
ax.scatter(X_train_norm[y_train==1, 0], X_train_norm[y_train==1, 1], c='blue', label='Classe 1', s=30, alpha=0.7)
ax.set_xlabel('Feature 1 (normalizada)')
ax.set_ylabel('Feature 2 (normalizada)')
ax.set_title('Decision Boundary - Logistic Regression')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.close()

print('O que observar: A boundary eh uma linha reta!')


---
## 7. Modelo 2: K-Nearest Neighbors (do Zero)

### O que observar:
- KNN eh um algoritmo lazy (nao treina, apenas memoriza)
- Simples mas pode ser custoso computacionalmente (O(n) por predicao)
- Sensivel a escala das features (ja normalizamos!)
- K eh critico: K=1 pode overfitar, K muito grande underfita

### O que concluir:
- KNN funciona bem para problemas nao-lineares
- K eh hiperparametro critico
- Requer dados em memoria (nao escalavel para datasets gigantes)

### Por que em ML:
KNN exemplifica como algoritmos diferentes podem resolver o mesmo problema.

### Conexao com intuicao:
KNN usa ideia intuitiva: "semelhante a seus vizinhos".


In [ ]:
class KNN:
    """K-Nearest Neighbors implementado do zero com NumPy.
    
    O que observar:
    - Nao treina, apenas armazena os dados
    - Distancia euclidiana mede proximidade
    - Votacao por maioria decide a classe
    """
    
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def treinar(self, X, y):
        """KNN nao treina, apenas armazena os dados.
        O que eh: Algoritmo lazy - toda computacao no teste
        """
        self.X_train = X
        self.y_train = y
        return self
    
    def distancia_euclidiana(self, x1, x2):
        """Calcula distancia euclidiana entre dois pontos.
        Formula: sqrt(sum((x1 - x2)^2))
        """
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def prever(self, X):
        """Preve classes usando os K vizinhos mais proximos.
        O que observar:
        1. Calcula distancia para todos os pontos de treinamento
        2. Seleciona K vizinhos mais proximos
        3. Vota pela classe mais frequente
        """
        predicoes = []
        
        for x_novo in X:
            # Calcular distancias para todos os pontos de treinamento
            distancias = np.array([self.distancia_euclidiana(x_novo, x_train) 
                                   for x_train in self.X_train])
            
            # Encontrar indices dos K vizinhos mais proximos
            k_indices = np.argsort(distancias)[:self.k]
            
            # Encontrar labels dos K vizinhos
            k_labels = self.y_train[k_indices]
            
            # Votacao por maioria
            labels_unicos, contagens = np.unique(k_labels, return_counts=True)
            predicao = labels_unicos[np.argmax(contagens)]
            
            predicoes.append(predicao)
        
        return np.array(predicoes).astype(int)

# Testar diferentes valores de K
resultados_k = {}
for k_valor in [1, 3, 5, 7, 9]:
    knn = KNN(k=k_valor)
    knn.treinar(X_train_norm, y_train)
    y_pred_train = knn.prever(X_train_norm)
    y_pred_test = knn.prever(X_test_norm)
    
    acc_train = accuracy(y_train, y_pred_train)
    acc_test = accuracy(y_test, y_pred_test)
    resultados_k[k_valor] = {'train': acc_train, 'test': acc_test}
    print(f'K={k_valor}: Train={acc_train:.4f}, Test={acc_test:.4f}')

print('\nO que observar: K=1 overfita (treino 100%), teste pior!')


In [ ]:
# Usar K=5 como padrao
knn = KNN(k=5)
knn.treinar(X_train_norm, y_train)
y_train_pred_knn = knn.prever(X_train_norm)
y_test_pred_knn = knn.prever(X_test_norm)

metricas_train_knn = calcular_metricas(y_train, y_train_pred_knn)
metricas_test_knn = calcular_metricas(y_test, y_test_pred_knn)

print('='*60)
print('KNN (K=5) - METRICAS')
print('='*60)
print(f'\nTREINO:')
print(f'  Accuracy:  {metricas_train_knn["accuracy"]:.4f}')
print(f'  Precision: {metricas_train_knn["precision"]:.4f}')
print(f'  Recall:    {metricas_train_knn["recall"]:.4f}')
print(f'  F1:        {metricas_train_knn["f1"]:.4f}')

print(f'\nTESTE:')
print(f'  Accuracy:  {metricas_test_knn["accuracy"]:.4f}')
print(f'  Precision: {metricas_test_knn["precision"]:.4f}')
print(f'  Recall:    {metricas_test_knn["recall"]:.4f}')
print(f'  F1:        {metricas_test_knn["f1"]:.4f}')

print(f'\nComparacao com Logistic Regression:')
print(f'LR teste accuracy: {metricas_test_lr["accuracy"]:.4f}')
print(f'KNN teste accuracy: {metricas_test_knn["accuracy"]:.4f}')


---
## 8. Comparacao: Decision Boundaries

### O que observar:
- LR cria boundaries lineares (reta no espaco 2D)
- KNN cria boundaries complexas/locais (forma irregular)
- Ambas predizem corretamente neste caso

### O que concluir:
- Nao-linearidade vs Simplicidade eh tradeoff
- Modelos diferentes sao melhores para problemas diferentes
- Dados linearmente separaveis: LR eh suficiente (mais simples!)

### Conexao com complexidade:
Modelos mais complexos nem sempre sao melhores (Occam's Razor).

### Por que em ML:
A escolha de modelo deve se basear em propriedades dos dados.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

h = 0.02
x_min, x_max = X_train_norm[:, 0].min() - 0.5, X_train_norm[:, 0].max() + 0.5
y_min, y_max = X_train_norm[:, 1].min() - 0.5, X_train_norm[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Dados
ax = axes[0]
ax.scatter(X_train_norm[y_train==0, 0], X_train_norm[y_train==0, 1], c='red', label='Classe 0', s=30, alpha=0.7)
ax.scatter(X_train_norm[y_train==1, 0], X_train_norm[y_train==1, 1], c='blue', label='Classe 1', s=30, alpha=0.7)
ax.set_title('Dados de Treino')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.grid(True, alpha=0.3)

# LR
ax = axes[1]
Z_lr = lr.prever(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z_lr, levels=1, alpha=0.3, colors=['red', 'blue'])
ax.scatter(X_train_norm[y_train==0, 0], X_train_norm[y_train==0, 1], c='red', label='Classe 0', s=30, alpha=0.7)
ax.scatter(X_train_norm[y_train==1, 0], X_train_norm[y_train==1, 1], c='blue', label='Classe 1', s=30, alpha=0.7)
ax.set_title(f'Logistic Regression (Linear, acc={metricas_test_lr["accuracy"]:.3f})')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.3)

# KNN
ax = axes[2]
Z_knn = knn.prever(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z_knn, levels=1, alpha=0.3, colors=['red', 'blue'])
ax.scatter(X_train_norm[y_train==0, 0], X_train_norm[y_train==0, 1], c='red', label='Classe 0', s=30, alpha=0.7)
ax.scatter(X_train_norm[y_train==1, 0], X_train_norm[y_train==1, 1], c='blue', label='Classe 1', s=30, alpha=0.7)
ax.set_title(f'KNN (K=5, Nao-linear, acc={metricas_test_knn["accuracy"]:.3f})')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.close()


---
## 9. Validacao Cruzada (Manual)

### O que observar:
- K-fold CV treina K modelos diferentes
- Cada fold serve como teste uma vez
- Reduz variancia da estimativa de performance
- Fornece intervalo de confianca (media +/- std)

### O que concluir:
- CV eh mais robusto que um unico train/test split
- Ideal para datasets pequenos
- Desvio padrao revela instabilidade do modelo

### Conexao com reproducibilidade:
CV reduz dependencia de um split aleatorio especifico.

### Por que em ML:
CV ajuda a detectar overfitting e instabilidade.



### O que observar sobre estabilidade do modelo
Cross-validation reduz variancia das estimativas de performance.
Desvio padrao alto indica instabilidade do modelo em diferentes splits.


In [ ]:
def k_fold_split(n_amostras, n_folds=5):
    """Cria indices para K-fold cross-validation.
    O que observar: Cada fold eh exclusivo como conjunto teste
    """
    indices = np.arange(n_amostras)
    np.random.shuffle(indices)
    fold_size = n_amostras // n_folds
    
    folds = []
    for i in range(n_folds):
        start = i * fold_size
        end = (i + 1) * fold_size
        test_indices = indices[start:end]
        train_indices = np.concatenate([indices[:start], indices[end:]])
        folds.append((train_indices, test_indices))
    
    return folds

# 5-fold cross-validation para LR
n_folds = 5
folds = k_fold_split(len(X), n_folds=n_folds)

scores_lr = []
scores_knn = []

for fold_idx, (train_idx, test_idx) in enumerate(folds):
    X_train_fold = X[train_idx]
    y_train_fold = y[train_idx]
    X_test_fold = X[test_idx]
    y_test_fold = y[test_idx]
    
    # Normalizar
    X_train_fold_norm, media, desvio = padronizar(X_train_fold)
    X_test_fold_norm, _, _ = padronizar(X_test_fold, media, desvio)
    
    # LR
    lr_fold = LogisticRegression(learning_rate=0.1, n_iteracoes=500)
    lr_fold.treinar(X_train_fold_norm, y_train_fold)
    y_pred_lr = lr_fold.prever(X_test_fold_norm)
    acc_lr = accuracy(y_test_fold, y_pred_lr)
    scores_lr.append(acc_lr)
    
    # KNN
    knn_fold = KNN(k=5)
    knn_fold.treinar(X_train_fold_norm, y_train_fold)
    y_pred_knn = knn_fold.prever(X_test_fold_norm)
    acc_knn = accuracy(y_test_fold, y_pred_knn)
    scores_knn.append(acc_knn)

print('='*60)
print(f'5-FOLD CROSS-VALIDATION')
print('='*60)
print(f'\nLogistic Regression:')
for i, score in enumerate(scores_lr):
    print(f'  Fold {i+1}: {score:.4f}')
print(f'  Media: {np.mean(scores_lr):.4f} (+/- {np.std(scores_lr):.4f})')

print(f'\nKNN (K=5):')
for i, score in enumerate(scores_knn):
    print(f'  Fold {i+1}: {score:.4f}')
print(f'  Media: {np.mean(scores_knn):.4f} (+/- {np.std(scores_knn):.4f})')


---
## 10. Tuning de Hiperparametros (Grid Search Manual)

### O que observar:
- Grid search testa todas as combinacoes
- Seleciona melhor combinacao por CV
- Pode ser custoso computacionalmente

### O que concluir:
- Hiperparametros controlam comportamento do modelo
- Tuning melhora performance significativamente
- Deve ser feito com CV, nao em teste!

### Por que em ML:
Nao ha valores "universais" de hiperparametros.



### Conexao com selecao de modelos
Grid search e validacao cruzada trabalham juntas para selecionar melhor modelo e hiperparametros.


In [ ]:
# ### TAREFA DO ALUNO-1: Grid Search para Logistic Regression
# Testar diferentes learning_rates e regularizacoes

learning_rates = [0.01, 0.05, 0.1]
regularizacoes = [0.0, 0.01, 0.1]

melhores_params = {}
melhor_score = 0

print('='*60)
print('GRID SEARCH - LOGISTIC REGRESSION')
print('='*60)

for lr_val in learning_rates:
    for reg_val in regularizacoes:
        scores = []
        
        for fold_idx, (train_idx, test_idx) in enumerate(folds):
            X_train_fold = X[train_idx]
            y_train_fold = y[train_idx]
            X_test_fold = X[test_idx]
            y_test_fold = y[test_idx]
            
            X_train_fold_norm, media, desvio = padronizar(X_train_fold)
            X_test_fold_norm, _, _ = padronizar(X_test_fold, media, desvio)
            
            model = LogisticRegression(learning_rate=lr_val, n_iteracoes=500, regularizacao=reg_val)
            model.treinar(X_train_fold_norm, y_train_fold)
            y_pred = model.prever(X_test_fold_norm)
            acc = accuracy(y_test_fold, y_pred)
            scores.append(acc)
        
        media_score = np.mean(scores)
        std_score = np.std(scores)
        
        if media_score > melhor_score:
            melhor_score = media_score
            melhores_params = {'lr': lr_val, 'reg': reg_val}
        
        print(f'LR={lr_val}, REG={reg_val}: {media_score:.4f} (+/- {std_score:.4f})')

print(f'\nMelhores parametros: LR={melhores_params["lr"]}, REG={melhores_params["reg"]}')
print(f'Melhor score: {melhor_score:.4f}')


In [ ]:
# ### TAREFA DO ALUNO-2: Grid Search para KNN
# Testar diferentes valores de K

k_valores = [1, 3, 5, 7, 9, 11]

melhores_params_knn = {}
melhor_score_knn = 0

print('='*60)
print('GRID SEARCH - KNN')
print('='*60)

for k_val in k_valores:
    scores = []
    
    for fold_idx, (train_idx, test_idx) in enumerate(folds):
        X_train_fold = X[train_idx]
        y_train_fold = y[train_idx]
        X_test_fold = X[test_idx]
        y_test_fold = y[test_idx]
        
        X_train_fold_norm, media, desvio = padronizar(X_train_fold)
        X_test_fold_norm, _, _ = padronizar(X_test_fold, media, desvio)
        
        model_knn = KNN(k=k_val)
        model_knn.treinar(X_train_fold_norm, y_train_fold)
        y_pred_knn = model_knn.prever(X_test_fold_norm)
        acc_knn = accuracy(y_test_fold, y_pred_knn)
        scores.append(acc_knn)
    
    media_score = np.mean(scores)
    std_score = np.std(scores)
    
    if media_score > melhor_score_knn:
        melhor_score_knn = media_score
        melhores_params_knn = {'k': k_val}
    
    print(f'K={k_val}: {media_score:.4f} (+/- {std_score:.4f})')

print(f'\nMelhores parametros KNN: K={melhores_params_knn["k"]}')
print(f'Melhor score KNN: {melhor_score_knn:.4f}')


In [ ]:
# ### TAREFA DO ALUNO-3: Implementar Distance Weighting para KNN
# Modificar KNN para usar pesos baseados em distancia
# Exemplo: vizinhos mais proximos tem peso maior na votacao
# Isto eh deixado como exercicio para o estudante

print('TAREFA DO ALUNO-3: Implementar KNN com distance weighting')
print('Dica: Use 1/distancia como peso na votacao')
print('Resultado esperado: melhor performance em dados complexos')


---
## 11. Analise de Erros Comuns

### Erro 1: Nao Normalizar


In [ ]:
# Demonstrar efeito de nao normalizar
knn_sem_norm = KNN(k=5)
knn_sem_norm.treinar(X_train, y_train)  # SEM normalizacao
y_pred_sem_norm = knn_sem_norm.prever(X_test)
acc_sem_norm = accuracy(y_test, y_pred_sem_norm)

knn_com_norm = KNN(k=5)
knn_com_norm.treinar(X_train_norm, y_train)
y_pred_com_norm = knn_com_norm.prever(X_test_norm)
acc_com_norm = accuracy(y_test, y_pred_com_norm)

print('### Erro 1: Nao Normalizar')
print(f'KNN SEM normalizacao: {acc_sem_norm:.4f}')
print(f'KNN COM normalizacao: {acc_com_norm:.4f}')
print(f'Diferenca: {acc_com_norm - acc_sem_norm:.4f}')
print('\nO que concluir: Normalizacao eh CRITICA para KNN!')


### Erro 2: Data Leakage no Preprocessing



### O que concluir sobre preprocessing
Data leakage invalida completamente a validacao do modelo.
E impossivel estimar performance real com dados vazando entre treino e teste.


In [ ]:
# Demonstrar data leakage
print('### Erro 2: Data Leakage')
print('\nERRADO (usar parametros globais):')
X_norm_global, media_global, desvio_global = padronizar(X)
X_train_leak = X_norm_global[:len(X_train)]
X_test_leak = X_norm_global[len(X_train):]
lr_leak = LogisticRegression(learning_rate=0.1, n_iteracoes=500)
lr_leak.treinar(X_train_leak, y_train)
acc_leak = accuracy(y_test, lr_leak.prever(X_test_leak))
print(f'Accuracy (COM data leakage): {acc_leak:.4f}')

print('\nCERTO (usar parametros do treino para teste):')
acc_correto = metricas_test_lr['accuracy']
print(f'Accuracy (SEM data leakage): {acc_correto:.4f}')
print('\nO que concluir: Sempre fit preprocessor no treino apenas!')


### Erro 3: Nao Validar Adequadamente


In [ ]:
print('### Erro 3: Nao Validar Adequadamente')
print('\nUsando unico train/test split:')
print(f'LR accuracy no teste unico: {metricas_test_lr["accuracy"]:.4f}')
print('\nUsando 5-fold CV:')
print(f'LR media CV: {np.mean(scores_lr):.4f}')
print(f'LR std CV: {np.std(scores_lr):.4f}')
print('\nO que concluir: CV eh mais robusto que unico split!')


### Erro 4: Overfitting com K=1


In [ ]:
print('### Erro 4: Overfitting com K=1 no KNN')
knn_k1 = KNN(k=1)
knn_k1.treinar(X_train_norm, y_train)
acc_train_k1 = accuracy(y_train, knn_k1.prever(X_train_norm))
acc_test_k1 = accuracy(y_test, knn_k1.prever(X_test_norm))
print(f'K=1 treino: {acc_train_k1:.4f}')
print(f'K=1 teste: {acc_test_k1:.4f}')
print(f'Diferenca: {acc_train_k1 - acc_test_k1:.4f}')
print('\nO que concluir: K=1 memoriza treino (100% no treino, pior no teste)')


### Erro 5: Usar modelo nao treinado


In [ ]:
print('### Erro 5: Usar modelo nao treinado')
lr_nao_treinado = LogisticRegression(learning_rate=0.1, n_iteracoes=500)
# Nao treinou! Pesos sao None
try:
    y_pred_erro = lr_nao_treinado.prever(X_test_norm)
except:
    print('Erro capturado: Nao pode usar modelo nao treinado!')
    print('\nO que concluir: Sempre treinar antes de prever!')


---
## 12. Resumo e Conclusoes

### Hierarquia de Classificacao

1. **Dataset**: Gerar e entender os dados
   - Explorar distribuicoes e visualizar
   - Identificar desbalanceamentos e outliers
   - Entender a natureza do problema

2. **Preprocessamento**: Preparar dados para ML
   - Normalizar/Padronizar features
   - Fazer split treino/teste antes de preprocessar
   - Evitar data leakage em todas as etapas

3. **Modelos**: Escolher e implementar
   - Logistic Regression (linear, simples, interpretavel)
   - KNN (nao-linear, local, sem treinamento)
   - Comparar propriedades e usar Occam's Razor

4. **Validacao**: Avaliar robustez
   - Metricas: Accuracy, Precision, Recall, F1
   - Confusion Matrix para entender erros
   - Cross-validation para validar reproducibilidade

5. **Tuning**: Otimizar hiperparametros
   - Grid search com cross-validation
   - Escolher melhor combinacao por media CV
   - Validar em teste final

### Checklist Final

- [ ] Dataset foi explorado adequadamente
- [ ] Preprocessamento foi aplicado corretamente
- [ ] Treino e teste foram separados ANTES do preprocessamento
- [ ] Pelo menos 2 modelos foram testados
- [ ] Metricas foram calculadas e interpretadas
- [ ] Cross-validation foi realizada corretamente
- [ ] Hiperparametros foram sintonizados com grid search
- [ ] Resultados foram visualizados
- [ ] Conclusoes foram documentadas

### Proximos Passos

1. Testar mais modelos (Decision Trees, SVM, etc.)
2. Trabalhar com datasets reais (Iris, MNIST, etc.)
3. Lidar com dados desbalanceados (SMOTE, class weights)
4. Implementar redes neurais (MLP)
5. Deploy do modelo em producao
6. Monitoramento e atualizacao continua


In [ ]:
# Definir variaveis do grid search (caso exercicio nao tenha sido executado)
if 'melhores_params' not in dir():
    melhores_params = {'lr': 0.1, 'epochs': 200}
if 'melhores_params_knn' not in dir():
    melhores_params_knn = {'k': 5}

print('='*60)
print('COMPARATIVO FINAL DOS MODELOS')
print('='*60)
print(f'\nLogistic Regression (teste):')
print(f'  Accuracy:  {metricas_test_lr["accuracy"]:.4f}')
print(f'  F1 Score:  {metricas_test_lr["f1"]:.4f}')

print(f'\nKNN K=5 (teste):')
print(f'  Accuracy:  {metricas_test_knn["accuracy"]:.4f}')
print(f'  F1 Score:  {metricas_test_knn["f1"]:.4f}')

print(f'\nCross-Validation (5-fold):')  
print(f'  LR media:  {np.mean(scores_lr):.4f} (+/- {np.std(scores_lr):.4f})')
print(f'  KNN media: {np.mean(scores_knn):.4f} (+/- {np.std(scores_knn):.4f})')

print(f'\nGrid Search Best:') 
print(f'  LR params: {melhores_params}')
print(f'  KNN params: {melhores_params_knn}')

print('\n' + '='*60)
print('FIM DO TUTORIAL - PARABENS!')
print('='*60)


---
## 13. Solucoes (Self-contained)

### SOLUCAO-1: Comparar diferentes valores de K


In [ ]:
# SOLUCAO-1: Comparacao de K values com visualizacao
import matplotlib.pyplot as plt

k_range = range(1, 20, 2)
train_scores = []
test_scores = []

for k_val in k_range:
    model = KNN(k=k_val)
    model.treinar(X_train_norm, y_train)
    train_acc = accuracy(y_train, model.prever(X_train_norm))
    test_acc = accuracy(y_test, model.prever(X_test_norm))
    train_scores.append(train_acc)
    test_scores.append(test_acc)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), train_scores, marker='o', label='Treino', linewidth=2)
ax.plot(list(k_range), test_scores, marker='s', label='Teste', linewidth=2)
ax.set_xlabel('K')
ax.set_ylabel('Accuracy')
ax.set_title('Efeito de K no KNN - Curva de Aprendizado')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.close()

melhor_k = list(k_range)[np.argmax(test_scores)]
print(f'Melhor K: {melhor_k} com accuracy={max(test_scores):.4f}')
print('O que observar: K maior = boundary mais suave, menos overfitting')


### SOLUCAO-2: Implementar Gradient Descent Estocastico


In [ ]:
# SOLUCAO-2: SGD vs Batch Gradient Descent - Comparacao
class LogisticRegressionSGD:
    """Logistic Regression com SGD (atualiza a cada exemplo ou mini-batch).
    
    O que observar: SGD eh mais ruidoso (ruido util para escapar de minimos locais)
    O que concluir: Trade-off entre velocidade (SGD) e estabilidade (BGD)
    """
    
    def __init__(self, learning_rate=0.01, n_iteracoes=100, batch_size=1):
        self.learning_rate = learning_rate
        self.n_iteracoes = n_iteracoes
        self.batch_size = batch_size
        self.pesos = None
        self.bias = None
        self.historico_loss = []
    
    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def treinar(self, X, y):
        m, n = X.shape
        self.pesos = np.zeros(n)
        self.bias = 0
        
        for iteracao in range(self.n_iteracoes):
            indices = np.random.permutation(m)
            for i in range(0, m, self.batch_size):
                batch_idx = indices[i:i+self.batch_size]
                X_batch = X[batch_idx]
                y_batch = y[batch_idx]
                
                z = np.dot(X_batch, self.pesos) + self.bias
                y_pred = self.sigmoid(z)
                
                erro = y_pred - y_batch
                self.pesos -= self.learning_rate * np.dot(X_batch.T, erro) / len(y_batch)
                self.bias -= self.learning_rate * np.mean(erro)
            
            z = np.dot(X, self.pesos) + self.bias
            y_pred = self.sigmoid(z)
            loss = -np.mean(y * np.log(np.clip(y_pred, 1e-15, 1-1e-15)) + (1-y) * np.log(np.clip(1-y_pred, 1e-15, 1-1e-15)))
            self.historico_loss.append(loss)
        
        return self
    
    def prever(self, X, threshold=0.5):
        z = np.dot(X, self.pesos) + self.bias
        return (self.sigmoid(z) >= threshold).astype(int)

# Comparar SGD vs BGD
lr_sgd = LogisticRegressionSGD(learning_rate=0.1, n_iteracoes=100, batch_size=1)
lr_sgd.treinar(X_train_norm, y_train)
acc_sgd = accuracy(y_test, lr_sgd.prever(X_test_norm))

print(f'SGD Accuracy: {acc_sgd:.4f}')
print(f'BGD Accuracy: {metricas_test_lr["accuracy"]:.4f}')
print('\nO que observar: Loss SGD eh mais ruidoso (oscila mais)')


### SOLUCAO-3: Implementar Voting Classifier


In [ ]:
# SOLUCAO-3: Ensemble simples (Voting) - Combinacao de modelos
class VotingClassifier:
    """Combina multiplos modelos por votacao por maioria.
    
    O que observar: Ensemble pode ser melhor que modelos individuais
    O que concluir: Diversidade de modelos eh importante para bom ensemble
    Conexao com mundo real: Muitos sistemas produtivos usam ensembles
    """
    
    def __init__(self, modelos):
        self.modelos = modelos
    
    def prever(self, X):
        predicoes = np.array([modelo.prever(X) for modelo in self.modelos])
        # Votacao por maioria
        resultado = []
        for j in range(X.shape[0]):
            votos = predicoes[:, j]
            labels_unicos, contagens = np.unique(votos, return_counts=True)
            resultado.append(labels_unicos[np.argmax(contagens)])
        return np.array(resultado).astype(int)

# Criar ensemble com modelos treinados
lr_ensemble = LogisticRegression(learning_rate=0.1, n_iteracoes=500)
lr_ensemble.treinar(X_train_norm, y_train)

knn_ensemble = KNN(k=5)
knn_ensemble.treinar(X_train_norm, y_train)

voting = VotingClassifier([lr_ensemble, knn_ensemble])
y_pred_voting = voting.prever(X_test_norm)
acc_voting = accuracy(y_test, y_pred_voting)

print('Comparacao de Accuracies:')
print(f'LR Accuracy: {metricas_test_lr["accuracy"]:.4f}')
print(f'KNN Accuracy: {metricas_test_knn["accuracy"]:.4f}')
print(f'Voting Accuracy: {acc_voting:.4f}')
print(f'\nO que observar: Ensemble combina fatos de ambos os modelos!')
